# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

* **Unit of Analysis:** One row represents one unique anonymized content page (`content_id` / URL snapshot).
* **Observation Time Window:** Historical performance captured across 90-day (`clicks_90d`, `impressions_90d`) and 30-day (`clicks_last_30d`, `impressions_last_30d`) aggregate windows up to mid-panel snapshot dates (`month = 2026-03`).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field Name | Category | Description / Reason |
| :--- | :--- | :--- |
| `impressions_90d`, `clicks_90d`, `ctr`, `clean_position`, `content_age_days` | **Feature** | Historical performance signals available at decision time. |
| `target_is_decayed` | **Label** | Binary proxy (1 = `trend_direction == 'down'` & `ctr < median`, 0 = Healthy). |
| `content_id`, `client_id`, `content_type` | **Context** | Metadata for tracking, segmenting, and reporting without model training. |
| `trend_pct` | **Excluded** | Shares underlying math directly with label construction (causes severe target leakage). |
| *Future Post-Refresh Metrics* | **Excluded** | Post-event clicks/sessions are unobservable at decision time. |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
import pandas as pd
import numpy as np

# Load dataset slice
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Clean position gotcha
df['clean_position'] = df['avg_position'].replace(0, np.nan)

# --- QUERY 1: Grain Verification ---
print("--- QUERY 1: GRAIN VERIFICATION ---")
total_rows = len(df)
unique_content = df['content_id'].nunique()
print(f"Total Rows: {total_rows}")
print(f"Unique Content IDs: {unique_content}")
print(f"Grain Check Passed (1 row = 1 content_id): {total_rows == unique_content}")

# --- QUERY 2: Row Counts & Data Dimensions ---
print("\n--- QUERY 2: ROW COUNTS & SPAN ---")
print(f"Total Rows Analyzed: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")

# --- QUERY 3: Availability Check (IS TRUE Filter & Missing Values) ---
print("\n--- QUERY 3: AVAILABILITY & MISSING VALUES ---")
df['is_available'] = (df['impressions_90d'] > 0) & (df['clean_position'].notna())
surviving_rows = df['is_available'].sum()

print(f"Rows with complete tracking (IS TRUE filter): {surviving_rows:,} / {len(df):,} ({surviving_rows/len(df)*100:.2f}%)")
print("\nMissing values in core feature set:")
print(df[['impressions_90d', 'clicks_90d', 'ctr', 'clean_position', 'content_age_days']].isna().sum())

--- QUERY 1: GRAIN VERIFICATION ---
Total Rows: 30000
Unique Content IDs: 30000
Grain Check Passed (1 row = 1 content_id): True

--- QUERY 2: ROW COUNTS & SPAN ---
Total Rows Analyzed: 30,000
Total Columns: 45

--- QUERY 3: AVAILABILITY & MISSING VALUES ---
Rows with complete tracking (IS TRUE filter): 28,795 / 30,000 (95.98%)

Missing values in core feature set:
impressions_90d        0
clicks_90d             0
ctr                    0
clean_position      1205
content_age_days       0
dtype: int64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

* **Unbalanced Historical Windows:** Aggregate metrics (`impressions_90d`) mask short-term volatility or sudden search demand spikes happening within the last 7 to 14 days.
* **GSC Position Averaging Gotcha:** `avg_position` relies on search result impressions; pages with zero impressions return position `0.0`, which requires explicitly converting to `NaN` to prevent biased position scores.
* **Lack of Qualitative Signals:** The data measures search volume drops directional trends, but cannot observe competitor content overhauls, algorithm update notes, or search intent shifts directly.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.